In [ ]:
# ==========================================
# CELL 1: INSTALLATION & ENVIRONMENT SETUP
# ==========================================

# Install Google ADK along with requested OpenTelemetry GCP extras and requests for handling API calls.
!pip install google-adk[otel-gcp]==1.30.0 requests

In [1]:
from typing import Optional, List
from pydantic import BaseModel, Field

class PipelineState(BaseModel):
    user_topic: str = Field(default="", description="The primary topic provided by the user.")
    raw_research: Optional[str] = Field(default=None, description="The initial unstructured facts collected by the researcher.")
    verification_report: Optional[str] = Field(default=None, description="Critique pointing out any missing context or accuracy errors.")
    refined_research: Optional[str] = Field(default=None, description="The verified and updated research data.")
    final_essay: Optional[str] = Field(default=None, description="The final polished article output.")

    # Target state list for your append function
    pipeline_history: List[str] = Field(default=[], description="Appended logs tracking state transition milestones.")

print("✅ PipelineState schema with list tracking initialized.")

✅ PipelineState schema with list tracking initialized.


In [41]:
from google.adk.agents.llm_agent import LlmAgent  # <-- Key Fix: Use LlmAgent instead of Agent
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.apps import App

# ==========================================
# DEFINE CUSTOM STATE LOGGING FUNCTION
# ==========================================
def append_to_state(tool_context, field: str, response: str) -> dict:
    """Appends a processing milestone string into a specified list field within the session state.

    Args:
        field: The key in the state dictionary.
        response: The logging milestone string to add.
    """
    existing_state = tool_context.state.get(field, [])
    if not isinstance(existing_state, list):
        existing_state = [existing_state] if existing_state else []

    tool_context.state[field] = existing_state + [response]
    return {"status": "success"}

# Step 1: Research (Reads from the user_topic set in the initial state)
researcher_agent = LlmAgent(
    model="gemini-3.5-flash",
    name="ResearcherAgent",
    instruction=(
        "You are a meticulous research analyst. Look at the value of the variable {user_topic} in the state. "
        "Provide exactly 3 key historical or technical milestones about it."
    ),
    output_key="raw_research"
)

# Step 2: Verify
verifier_subagent = LlmAgent(
    model="gemini-3.5-flash",
    name="VerifierAgent",
    instruction=(
        "You are a factual auditor. Look at the data stored in {raw_research}. "
        "Review it strictly for technical accuracy, potential omissions, or historical context. "
        "Output a short verification report identifying what needs adjustment or confirming it is solid."
    ),
    output_key="verification_report"
)

# Step 3: Refine
refiner_subagent = LlmAgent(
    model="gemini-3.5-flash",
    name="RefinerAgent",
    instruction=(
        "You are an editor. Look at the initial {raw_research} and combine it with the advice in {verification_report}. "
        "Rewrite the research notes so they are perfectly accurate, comprehensive, and clear."
    ),
    output_key="refined_research"
)

# Step 4: Write
writer_agent = LlmAgent(
    model="gemini-3.5-flash",
    name="WriterAgent",
    instruction=(
        "You are a professional author. Read the facts stored in the {refined_research} variable. "
        "Transform those finalized notes into an elegant 2-paragraph essay. Do not invent new facts."
    ),
    output_key="final_essay"
)

# ==========================================
# LOGGING AGENTS (Plain Python Tools Setup)
# ==========================================
research_logger = LlmAgent(
    model="gemini-3.5-flash",
    name="ResearchLogger",
    instruction="Invoke the append_to_state function tool to save 'Completed Raw Research Step' to pipeline_history.",
    tools=[append_to_state]
)

verification_logger = LlmAgent(
    model="gemini-3.5-flash",
    name="VerificationLogger",
    instruction="Invoke the append_to_state function tool to save 'Completed Verification Step' to pipeline_history.",
    tools=[append_to_state]
)

# ==========================================
# ASSEMBLE SEQUENTIAL WORKFLOW
# ==========================================
root_agent = SequentialAgent(
    name="StateHookPipeline",
    sub_agents=[
        researcher_agent,
        research_logger,
        verifier_subagent,
        verification_logger,
        refiner_subagent,
        writer_agent
    ],
    description="Sequential pipeline executing custom state logging functions natively during memory transits."
)


In [46]:
from IPython.display import display, Markdown

# Initialize the App with the required name and root agent
app = App(name="PipelineApp", root_agent=root_agent)

print("🚀 Starting Pipeline Execution...\n")

prompt = "The engineering marvel of the Golden Gate Bridge."
initial_state = PipelineState(user_topic=prompt)

dh = display(Markdown("⏳ *Running pipeline steps...*"), display_id=True)

# Fix: In ADK 1.30.0, execute the agent by calling it directly as a coroutine.
# This returns the updated state dictionary.
final_state = await app.root_agent(
    query=prompt,
    state=initial_state.model_dump()
)

# Format and display output logs
output_md = f"""
### 📋 Transition Verification Trace
* **User Topic State:** {final_state.get('user_topic')}

---

#### 🛠️ History Array Appended items (`pipeline_history`):
{final_state.get('pipeline_history', [])}

---

#### ✍️ Final Production Output:
{final_state.get('final_essay')}
"""

dh.update(Markdown(output_md))

🚀 Starting Pipeline Execution...



⏳ *Running pipeline steps...*

TypeError: 'SequentialAgent' object is not callable